<a href="https://colab.research.google.com/github/Anorwed/vkr/blob/main/qsar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =============================================================================
# 1. Установка зависимостей (магия Colab — надёжнее subprocess)
# =============================================================================
%pip install rdkit pandas numpy matplotlib scikit-learn -q

# Если RDKit не подхватился сразу — перезапускаем ядро и просим запустить ещё раз
import importlib.util
import sys
from IPython import get_ipython

if importlib.util.find_spec("rdkit") is None:
    print("⚠️  Зависимости установлены, но ядро нужно перезапустить.")
    print("   Перезапускаю… запустите эту же ячейку ещё раз (Ctrl+Enter).")
    get_ipython().kernel.do_shutdown(True)
    sys.exit(0)

# =============================================================================
# 2. Импорты
# =============================================================================
import os
import pickle
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Sequence, Tuple
import urllib.request

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, rdMolDescriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

# =============================================================================
# Config
# =============================================================================
SCRIPT_DIR = Path(os.getcwd())
OUTPUT_DATA = SCRIPT_DIR / "processed_data.csv"
OUTPUT_MODEL = SCRIPT_DIR / "model.pkl"
OUTPUT_METRICS = SCRIPT_DIR / "metrics.txt"
PLOTS_DIR = SCRIPT_DIR / "plots"

SMILES_COLUMN = "SMILES"
ACTIVITY_COLUMN = "Activity"

MORGAN_RADIUS = 2
MORGAN_N_BITS = 2048
MORGAN_PREFIX = "Morgan_"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_ESTIMATORS = 300
MIN_SAMPLES_PER_CLASS_FOR_SPLIT = 2

DESCRIPTOR_COLUMNS = [
    "MolWt",
    "MolLogP",
    "NumHDonors",
    "NumHAcceptors",
    "TPSA",
    "NumRotatableBonds",
]

CSV_URL = "https://raw.githubusercontent.com/Anorwed/vkr/main/demo_smiles.csv"
CSV_LOCAL = SCRIPT_DIR / "demo_smiles.csv"


@dataclass
class ModelResult:
    model: Optional[RandomForestClassifier]
    y_test: Optional[pd.Series]
    y_pred: Optional[np.ndarray]
    y_proba: Optional[np.ndarray]
    feature_names: List[str]
    metrics: dict


# =============================================================================
# Utils
# =============================================================================
def log(message: str) -> None:
    print(f"[QSAR] {message}")


def download_csv(url: str = CSV_URL, local_path: Path = CSV_LOCAL) -> Path:
    if not local_path.exists():
        log(f"Downloading dataset from {url}")
        urllib.request.urlretrieve(url, local_path)
        log(f"Saved to {local_path}")
    else:
        log(f"Using existing local file: {local_path}")
    return local_path


def load_data(csv_path: Path) -> pd.DataFrame:
    log(f"Loading data from {csv_path}")
    data = pd.read_csv(csv_path)
    if SMILES_COLUMN not in data.columns:
        raise ValueError(f"Input CSV must contain a '{SMILES_COLUMN}' column")
    return data


def canonical_smiles(mol: Chem.Mol) -> str:
    return Chem.MolToSmiles(mol, isomericSmiles=True)


def validate_smiles(data: pd.DataFrame) -> Tuple[pd.DataFrame, List[Chem.Mol]]:
    valid_rows = []
    valid_mols: List[Chem.Mol] = []
    seen_canonical = set()

    for _, row in data.iterrows():
        raw_smiles = row.get(SMILES_COLUMN)
        if pd.isna(raw_smiles):
            continue
        smiles = str(raw_smiles).strip()
        if not smiles:
            continue

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        canonical = canonical_smiles(mol)
        if canonical in seen_canonical:
            continue

        seen_canonical.add(canonical)
        cleaned_row = row.copy()
        cleaned_row[SMILES_COLUMN] = canonical
        valid_rows.append(cleaned_row)
        valid_mols.append(mol)

    cleaned = pd.DataFrame(valid_rows).reset_index(drop=True)
    return cleaned, valid_mols


def compute_descriptors(mols: Sequence[Chem.Mol]) -> pd.DataFrame:
    records = []
    for mol in mols:
        records.append(
            {
                "MolWt": Descriptors.MolWt(mol),
                "MolLogP": Crippen.MolLogP(mol),
                "NumHDonors": Lipinski.NumHDonors(mol),
                "NumHAcceptors": Lipinski.NumHAcceptors(mol),
                "TPSA": rdMolDescriptors.CalcTPSA(mol),
                "NumRotatableBonds": Lipinski.NumRotatableBonds(mol),
            }
        )
    return pd.DataFrame(records, columns=DESCRIPTOR_COLUMNS)


def compute_fingerprints(
    mols: Sequence[Chem.Mol],
    radius: int = MORGAN_RADIUS,
    n_bits: int = MORGAN_N_BITS,
) -> pd.DataFrame:
    fingerprint_rows = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for row_index, mol in enumerate(mols):
        bit_vector = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(bit_vector, fingerprint_rows[row_index])

    columns = [f"{MORGAN_PREFIX}{bit_index}" for bit_index in range(n_bits)]
    return pd.DataFrame(fingerprint_rows, columns=columns)


def prepare_activity(activity: pd.Series) -> pd.Series:
    non_null = activity.dropna()
    if non_null.empty:
        raise ValueError("Activity column is empty")

    if pd.api.types.is_numeric_dtype(non_null):
        labels = activity.astype("Int64")
    else:
        codes, uniques = pd.factorize(activity.astype(str), sort=True)
        if len(uniques) > 2:
            raise ValueError("Activity column must contain at most two classes")
        labels = pd.Series(codes, index=activity.index, dtype="int64")

    unique_labels = sorted(pd.Series(labels).dropna().unique().tolist())
    if len(unique_labels) != 2:
        raise ValueError("Binary classification requires exactly two Activity classes")

    mapping = {label: integer for integer, label in enumerate(unique_labels)}
    return pd.Series(labels.map(mapping), index=activity.index, dtype="int64")


def build_feature_matrix(processed: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    fingerprint_columns = [c for c in processed.columns if c.startswith(MORGAN_PREFIX)]
    feature_names = DESCRIPTOR_COLUMNS + fingerprint_columns
    return processed[feature_names].copy(), feature_names


# =============================================================================
# ML
# =============================================================================
def train_model(processed: pd.DataFrame) -> ModelResult:
    if ACTIVITY_COLUMN not in processed.columns:
        return ModelResult(
            model=None, y_test=None, y_pred=None, y_proba=None,
            feature_names=[], metrics={"status": "skipped", "reason": "Activity column not found"}
        )

    labeled = processed.dropna(subset=[ACTIVITY_COLUMN]).copy()
    if labeled.empty:
        return ModelResult(
            model=None, y_test=None, y_pred=None, y_proba=None,
            feature_names=[], metrics={"status": "skipped", "reason": "Activity column is empty"}
        )

    try:
        y = prepare_activity(labeled[ACTIVITY_COLUMN])
    except ValueError as exc:
        return ModelResult(
            model=None, y_test=None, y_pred=None, y_proba=None,
            feature_names=[], metrics={"status": "skipped", "reason": str(exc)}
        )

    X, feature_names = build_feature_matrix(labeled)
    class_counts = y.value_counts()
    if len(y) < 5 or len(class_counts) != 2:
        return ModelResult(
            model=None, y_test=None, y_pred=None, y_proba=None,
            feature_names=feature_names, metrics={"status": "skipped", "reason": "Not enough labeled samples"}
        )

    stratify = y if class_counts.min() >= MIN_SAMPLES_PER_CLASS_FOR_SPLIT else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
    )

    model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    return ModelResult(
        model=model, y_test=y_test, y_pred=y_pred, y_proba=y_proba,
        feature_names=feature_names, metrics=evaluate_model(y_test, y_pred, y_proba)
    )


def evaluate_model(y_test: pd.Series, y_pred: np.ndarray, y_proba: Optional[np.ndarray] = None) -> dict:
    metrics = {
        "status": "trained",
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }
    if y_proba is not None and len(np.unique(y_test)) == 2:
        try:
            metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        except ValueError as exc:
            metrics["roc_auc"] = f"not available: {exc}"
    else:
        metrics["roc_auc"] = "not available"
    return metrics


def save_model(model_result: ModelResult, output_path: Path = OUTPUT_MODEL) -> None:
    if model_result.model is None:
        log("Model was not trained; model.pkl will not be written")
        return
    with output_path.open("wb") as handle:
        pickle.dump({
            "model": model_result.model,
            "feature_names": model_result.feature_names,
            "descriptor_columns": DESCRIPTOR_COLUMNS,
            "fingerprint_radius": MORGAN_RADIUS,
            "fingerprint_bits": MORGAN_N_BITS,
        }, handle)
    log(f"Saved trained model to {output_path}")


def save_metrics(model_result: ModelResult, output_path: Path = OUTPUT_METRICS) -> None:
    lines = ["QSAR AutoDescriptor Pipeline metrics", ""]
    for key, value in model_result.metrics.items():
        lines.append(f"{key}: {value}")
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    log(f"Saved metrics to {output_path}")


# =============================================================================
# Plots
# =============================================================================
def ensure_plots_dir() -> None:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def plot_distributions(processed: pd.DataFrame) -> None:
    ensure_plots_dir()
    specs = [
        ("MolWt", "Molecular Weight Distribution", "Molecular Weight", "hist_molwt.png"),
        ("MolLogP", "LogP Distribution", "MolLogP", "hist_logp.png"),
    ]
    for column, title, xlabel, filename in specs:
        plt.figure(figsize=(8, 5))
        plt.hist(processed[column].dropna(), bins=30, color="#4c72b0", edgecolor="black")
        plt.title(title)
        plt.xlabel(xlabel)
        plt.ylabel("Count")
        plt.tight_layout()
        out = PLOTS_DIR / filename
        plt.savefig(out, dpi=150)
        plt.close()
        log(f"Saved plot {out}")


def plot_confusion_matrix(model_result: ModelResult) -> None:
    if model_result.y_test is None or model_result.y_pred is None:
        log("Confusion matrix plot skipped")
        return
    ensure_plots_dir()
    cm = confusion_matrix(model_result.y_test, model_result.y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    display.plot(cmap="Blues", values_format="d")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    out = PLOTS_DIR / "confusion_matrix.png"
    plt.savefig(out, dpi=150)
    plt.close()
    log(f"Saved plot {out}")


def plot_feature_importance(model_result: ModelResult, top_n: int = 20) -> None:
    if model_result.model is None:
        log("Feature importance plot skipped")
        return
    ensure_plots_dir()
    importances = pd.Series(
        model_result.model.feature_importances_,
        index=model_result.feature_names,
        dtype="float64",
    ).sort_values(ascending=False).head(top_n).sort_values()

    plt.figure(figsize=(9, 7))
    importances.plot(kind="barh", color="#55a868")
    plt.title(f"Top {len(importances)} RandomForest Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    out = PLOTS_DIR / "feature_importance.png"
    plt.savefig(out, dpi=150)
    plt.close()
    log(f"Saved plot {out}")


# =============================================================================
# Main
# =============================================================================
def run_pipeline() -> int:
    log("Starting QSAR AutoDescriptor Pipeline")
    try:
        csv_path = download_csv()
        log(f"Found CSV: {csv_path.name}")

        raw_data = load_data(csv_path)
        log(f"Molecules before cleaning: {len(raw_data)}")

        cleaned_data, mols = validate_smiles(raw_data)
        log(f"Molecules after cleaning: {len(cleaned_data)}")
        if len(cleaned_data) == 0:
            raise ValueError("No valid molecules remain after SMILES validation")

        log("Computing RDKit descriptors")
        descriptors = compute_descriptors(mols)

        log(f"Computing Morgan fingerprints (radius={MORGAN_RADIUS}, nBits={MORGAN_N_BITS})")
        fingerprints = compute_fingerprints(mols)

        processed = pd.concat(
            [cleaned_data.reset_index(drop=True), descriptors, fingerprints],
            axis=1,
        )
        processed.to_csv(OUTPUT_DATA, index=False)
        log(f"Saved processed data to {OUTPUT_DATA}")

        log("Building visualizations")
        plot_distributions(processed)

        log("Training/evaluating model when Activity is available")
        model_result = train_model(processed)
        save_metrics(model_result)
        save_model(model_result)
        plot_confusion_matrix(model_result)
        plot_feature_importance(model_result)

        log("Metrics:")
        for key, value in model_result.metrics.items():
            log(f"  {key}: {value}")

        log("Pipeline finished successfully")
        return 0
    except Exception as exc:
        print(f"[QSAR][ERROR] {exc}", file=sys.stderr)
        return 1


if __name__ == "__main__":
    raise SystemExit(run_pipeline())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 33.8 MB/s eta 0:00:00
[QSAR] Starting QSAR AutoDescriptor Pipeline
[QSAR] Downloading dataset from https://raw.githubusercontent.com/Anorwed/vkr/main/demo_smiles.csv
[QSAR] Saved to /content/demo_smiles.csv
[QSAR] Found CSV: demo_smiles.csv
[QSAR] Loading data from /content/demo_smiles.csv
[QSAR] Molecules before cleaning: 40
[QSAR] Molecules after cleaning: 40
[QSAR] Computing RDKit descriptors
[QSAR] Computing Morgan fingerprints (radius=2, nBits=2048)
[QSAR] Saved processed data to /content/processed_data.csv
[QSAR] Building visualizations


[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerator
[20:03:10] DEPRECATION WARNING: please use MorganGenerat

[QSAR] Saved plot /content/plots/hist_molwt.png
[QSAR] Saved plot /content/plots/hist_logp.png
[QSAR] Training/evaluating model when Activity is available
[QSAR] Saved metrics to /content/metrics.txt
[QSAR] Saved trained model to /content/model.pkl
[QSAR] Saved plot /content/plots/confusion_matrix.png
[QSAR] Saved plot /content/plots/feature_importance.png
[QSAR] Metrics:
[QSAR]   status: trained
[QSAR]   accuracy: 0.875
[QSAR]   confusion_matrix: [[2, 1], [0, 5]]
[QSAR]   roc_auc: 0.9333333333333333
[QSAR] Pipeline finished successfully


SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
